# Recovery and the decision paper

Test six recovery options against the forecast, find the combinations that bring the opening back, choose with the sponsor's priorities in mind, and write the one-page paper that asks for the decisions you need.

*From [CloudTech Academy](https://academy.cloudtechanalytics.com/learn/project-manager-capstone/recovery-and-the-decision-paper). Run the cells in order (Runtime → Run all), edit them, and experiment. Answer the practice questions on the lesson page.*

## The problem

You now have a status (behind and over budget), a forecast (19 to 26 March, over by ₦17m to ₦30m), a priced risk register and a view of six change requests. The sponsor has one more question: **what should we do?** Six recovery options are on the table, and they interact. Some save days on the critical chain; some save none; some can be combined; one needs the medical director's agreement and changes what the laboratory opens with.

A recovery plan isn't "work harder". It's a short list of actions, each with a cost and a measured effect on the date, plus a clear account of what the sponsor must decide and by when.

## The concept

### Three ways to recover time

| Technique | What it means | Here |
| :-- | :-- | :-- |
| **Crashing** | Pay more to shorten a task | Air freight for the analysers; weekend shifts |
| **Fast-tracking** | Do things in parallel that were planned in sequence | Stock reagents while the analysers are installed |
| **Reducing scope** | Do less, or defer part of it | Open with the most-requested tests; validate the rest after opening |

All of them cost something: money, risk, or quality. And all of them only help on the **critical chain**.

### Choosing

Compare options on:

1. **Days saved** on the forecast, measured by re-running the schedule, not read from the proposal.
2. **Extra cost.**
3. **What it puts at risk:** quality, safety, the full test menu, the regulator's trust.
4. **Whether the sponsor's priorities allow it.** Here the managing director wants the date; the medical director won't open without the full menu; the CFO won't be surprised again.

![Three ways to recover time, invented options compared by claimed and really saved days, and a trigger agreed in advance](https://academy.cloudtechanalytics.com/images/courses/pm-capstone/recovery-options.svg "Test each option; combine; hold one back behind a trigger.")

Where no combination is safe, say so, and say what you'd do if the next problem arrives. A **trigger** ("if clearance isn't complete by 20 January, we use the reduced menu") turns a worry into a plan.

### The decision paper

One page for the sponsor, in this order:

1. **The answer:** status, forecast and what you recommend, in two sentences.
2. **What you need decided, and by when.**
3. **Evidence:** the three numbers that matter (forecast date, cost, risk).
4. **Options considered**, with cost and days saved, and why the others were rejected.
5. **Risks and triggers.**
6. **What you'll report next, and when.**

![The six parts of a one-page decision paper for a sponsor](https://academy.cloudtechanalytics.com/images/courses/pm-capstone/decision-paper.svg "Recommendation first; decisions with dates; three numbers; options; trigger.")

> **Note:** > A good decision paper leads with the decision, and makes the unpopular parts impossible to miss: the date is gone, the budget is higher, and one of the sponsor's own requests should wait.

## Example

Set up the forecast as in lesson 6, then apply each recovery option to the remaining-work schedule:

In [ ]:
import itertools
import numpy as np
import pandas as pd

base = "https://academy.cloudtechanalytics.com/datasets/lab/"
tasks = pd.read_csv(base + "tasks.csv").fillna({"predecessors": ""})
status = pd.read_csv(base + "weekly_status.csv")
options = pd.read_csv(base + "options.csv")
START = np.datetime64("2026-11-02")
PROMISE_DAY = 91

def schedule(tasks, durations):
    preds = {t: [p for p in ps.split(";") if p] for t, ps in zip(tasks["task_id"], tasks["predecessors"])}
    ef = {}
    for t in tasks["task_id"]:
        ef[t] = max((ef[p] for p in preds[t]), default=0) + durations[t]
    return max(ef.values())

def finish_date(days):
    return np.busday_offset(START, int(np.ceil(days)) - 1, roll="forward")

tasks["budget"] = tasks["likely_days"] * tasks["daily_cost_ngn"]
now = status[status["week"] == 10].set_index("task_id")
first_week = status.groupby("task_id")["week"].min()
BAC = tasks["budget"].sum()
EV = (now["percent_complete"] / 100 * tasks.set_index("task_id").loc[now.index, "budget"]).sum()
AC = now["actual_cost_ngn"].sum()
typical = BAC / (EV / AC)

rem = {}
for t, likely in zip(tasks["task_id"], tasks["likely_days"]):
    if t not in now.index:
        rem[t] = likely
        continue
    pc = now.loc[t, "percent_complete"] / 100
    elapsed = 5 * (10 - first_week[t] + 1)
    rem[t] = 0 if pc >= 1 else elapsed * (1 - pc) / pc        # the pace achieved so far

forecast = 50 + schedule(tasks, rem)
print(f"Forecast with no action: day {forecast:.1f}, {finish_date(forecast)}")
print(options[["option_id", "option", "extra_cost_ngn", "days_saved"]].to_string(index=False))

*Expected output:*

```
Forecast with no action: day 104.7, 2027-03-26
option_id                             option  extra_cost_ngn  days_saved
       O1                Carry on as planned               0           0
       O2               Fly the analysers in         5500000           9
       O3     Weekend working on the fit-out         3200000           5
       O4       Start reagent stocking early          450000           4
       O5      Open with a reduced test menu         1200000           6
       O6 Add a molecular testing room first        16000000         -14
```

An option's "days saved" is what the proposer claims, and it only counts if it lands on the critical chain. Link each option to the task it changes, apply it, and **measure**:

In [ ]:
applies_to = {"O2": "B4", "O3": "B2", "O4": "E2", "O5": "F1"}      # O1 changes nothing; O6 adds scope

def finish_with(chosen):
    r = dict(rem)
    for o in chosen:
        days = int(options.loc[options["option_id"] == o, "days_saved"].iloc[0])
        t = applies_to[o]
        r[t] = max(0, r[t] - days)
    return 50 + schedule(tasks, r)

for o in applies_to:
    gain = forecast - finish_with([o])
    print(f"{o}: claims {int(options.loc[options['option_id'] == o, 'days_saved'].iloc[0])} days, really saves {gain:.1f}")

*Expected output:*

```
O2: claims 9 days, really saves 9.0
O3: claims 5 days, really saves 0.0
O4: claims 4 days, really saves 4.0
O5: claims 6 days, really saves 6.0
```

**O3**, weekend shifts on the fit-out, claims 5 days and saves none, as in lesson 7. The other three are real. Now try every combination of the four recovery options:

In [ ]:
cost = dict(zip(options["option_id"], options["extra_cost_ngn"]))
rows = []
for n in range(0, 5):
    for combo in itertools.combinations(applies_to, n):
        end = finish_with(combo)
        rows.append((" + ".join(combo) or "none", round(end, 1), str(finish_date(end)), sum(cost[o] for o in combo)))
table = pd.DataFrame(rows, columns=["options", "finish_day", "date", "extra_cost_ngn"]).sort_values(["finish_day", "extra_cost_ngn"])
print(table.drop_duplicates("finish_day").head(6).to_string(index=False))   # the cheapest way to each finish day

*Expected output:*

```
options  finish_day       date  extra_cost_ngn
O2 + O4 + O5        85.7 2027-03-01         7150000
     O2 + O5        89.7 2027-03-05         6700000
     O2 + O4        91.7 2027-03-09         5950000
     O4 + O5        94.7 2027-03-12         1650000
          O2        95.7 2027-03-15         5500000
          O5        98.7 2027-03-18         1200000
```

The cheapest way to meet the promised day (91) is **O2 + O5**: air freight and a reduced opening menu, for ₦6.7m, opening on 5 March. Adding **O4** takes it back to 1 March, for ₦0.45m more. **O2 + O4** alone, with the full test menu, opens on **Tuesday 9 March**, one working day after the promise, for ₦5.95m.

That's the real decision. Meeting the promise to the day needs the reduced menu, which the medical director has said he won't accept without a clinical case; the full menu costs one day, and the sponsor has to decide whether to take that to the commissioner.

And none of those dates includes the risk register. The expected delay on the critical chain is 13.9 days (lesson 7), so a plan that opens on 9 March with no allowance is a plan with perhaps a 20% chance of doing so. The honest recommendation has three parts:

1. **Approve O2 and O4 now.** They keep the full menu, cost ₦5.95m, and bring the forecast to 9 March.
2. **Keep O5 as a trigger, not a plan.** If the analysers have not cleared customs by a named date, use the reduced menu to protect the opening, with the medical director's agreement settled in advance.
3. **Tell the commissioner now** that the opening will be in the week of 8 March, with a realistic range, instead of letting the date go quietly wrong.

The cost position follows:

In [ ]:
chosen = ["O2", "O4"]
extra = sum(cost[o] for o in chosen)
atypical = AC + (BAC - EV)
print(f"Approved budget:           ₦{BAC:,.0f}")
print(f"Forecast with recovery:    ₦{atypical + extra:,.0f} to ₦{typical + extra:,.0f}")
print(f"Opening with recovery:     {finish_date(finish_with(chosen))}")

*Expected output:*

```
Approved budget:           ₦90,750,000
Forecast with recovery:    ₦113,874,000 to ₦126,287,860
Opening with recovery:     2027-03-09
```

## Walkthrough

1. Rebuild the forecast with no action and note its day and date.
2. Link each recovery option to the task it changes, and **measure** its real effect.
3. Try every combination, and list finish date and cost.
4. Mark which combinations meet the promised day, and what each one puts at risk.
5. Choose, with reasons, and name a **trigger** for the option you're holding back.
6. Calculate the revised cost range.
7. Write the decision paper (the first task below), and the answers to the sponsor's hardest questions (the second).

## Practice

**Practice:** If **O2 and O4** are both approved, on what date does the project finish? Type it as YYYY-MM-DD.

*Work it out below, then enter your answer on the lesson page.*

In [ ]:
# Your code here

**Practice:** What is the **cheapest combination** of the four recovery options (O2 to O5) that finishes **by the promised day** (day 91)? Give its **extra cost** in naira.

*Work it out below, then enter your answer on the lesson page.*

In [ ]:
# Your code here

**Practice:** How many of the four recovery options O2 to O5 **actually shorten the finish** when applied alone?

*Work it out below, then enter your answer on the lesson page.*

In [ ]:
# Your code here

**Practice:** With **O2 and O4** approved, what is the **upper end of the cost forecast** (typical EAC plus the options' cost), to the nearest naira?

*Work it out below, then enter your answer on the lesson page.*

In [ ]:
# Your code here

**Task:** Write the **decision paper** for the sponsor (150 to 280 words), in this order: the **answer and recommendation** first, the **decisions you need** and when, the **evidence** (forecast date, cost and risk), the **options** considered and why the others were rejected, and the **trigger** for the contingency plan.

*Work it out below, then enter your answer on the lesson page.*

In [ ]:
# Your code here

**Task:** Write the answers to the sponsor's **three hardest questions** (100 to 200 words in all): "Why can't we just keep 8 March?", "Why not approve the weekend working?" and "Why shouldn't I add the molecular room now?"

*Work it out below, then enter your answer on the lesson page.*

In [ ]:
# Your code here

## Check your understanding